In [1]:
!nvidia-smi

Sat Nov  7 15:03:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU available:', torch.cuda.is_available())

if torch.cuda.is_available():
  print('GPU:', torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
CUDA: 13.0
GPU available: True
GPU: Tesla T4


Скачаем оригинальный код:

In [3]:
!git clone https://github.com/southerly7/Proposed.git
%cd Proposed

!git rev-parse HEAD
!git status --short

Cloning into 'Proposed'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 52 (delta 26), reused 30 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 31.84 KiB | 4.55 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/Proposed
0c57846d62c25ac548d7810d425f00b9299a8e54


In [4]:
import sys
import torchvision

print('Python:', sys.version)
print('Torchvision:', torchvision.__version__)

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Torchvision: 0.29.1+cu130


In [5]:
%pip install timm tensorboard

In [6]:
import timm

print('timm:', timm.__version__)

timm: 1.0.31


In [7]:
!python main.py --help

Traceback (most recent call last):
  File "/content/Proposed/main.py", line 18, in <module>
    from optim_factory import create_optimizer, LayerDecayValueAssigner
  File "/content/Proposed/optim_factory.py", line 8, in <module>
    from timm.optim.nadam import Nadam
ImportError: cannot import name 'Nadam' from 'timm.optim.nadam' (/usr/local/lib/python3.13/dist-packages/timm/optim/nadam.py)


Из-за того что авторы используют более старую версию, у некоторых классов другие имена. Исправим:

In [8]:
from pathlib import Path

path = Path('/content/Proposed/optim_factory.py')
code = path.read_text()

old = 'from timm.optim.nadam import Nadam'
new = 'from timm.optim.nadam import NAdamLegacy as Nadam'

assert old in code, 'No import'
path.write_text(code.replace(old, new))

print('ok')

ok


In [9]:
%cd /content/Proposed
!python main.py --help

/content/Proposed
Traceback (most recent call last):
  File "/content/Proposed/main.py", line 18, in <module>
    from optim_factory import create_optimizer, LayerDecayValueAssigner
  File "/content/Proposed/optim_factory.py", line 9, in <module>
    from timm.optim.novograd import NovoGrad
ModuleNotFoundError: No module named 'timm.optim.novograd'


Аналогичная ситуация, но только в нашем эксперименте вообще не используется NovoGrad, так что этот импорт можем вообще убрать. И вообще уберем импорты оптимизаторов, которые не используем.


In [10]:
lines = code.splitlines()
lines = [line for line in lines if not line.startswith('from timm.optim.')]

code = '\n'.join(lines) + '\n'

path.write_text(code)

6740

In [11]:
%cd /content/Proposed
!git diff -- optim_factory.py

/content/Proposed
diff --git a/optim_factory.py b/optim_factory.py
index 0206af3..99c0a00 100644
--- a/optim_factory.py
+++ b/optim_factory.py
@@ -1,16 +1,6 @@
 import torch
 from torch import optim as optim
 
-from timm.optim.adafactor import Adafactor
-from timm.optim.adahessian import Adahessian
-from timm.optim.adamp import AdamP
-from timm.optim.lookahead import Lookahead
-from timm.optim.nadam import Nadam
-from timm.optim.novograd import NovoGrad
-from timm.optim.nvnovograd import NvNovoGrad
-from timm.optim.radam import RAdam
-from timm.optim.rmsprop_tf import RMSpropTF
-from timm.optim.sgdp import SGDP
 
 import json
 


In [12]:
%cd /content/Proposed
!python main.py --help

/content/Proposed
Traceback (most recent call last):
  File "/content/Proposed/main.py", line 21, in <module>
    from engine import train_one_epoch, evaluate
  File "/content/Proposed/engine.py", line 7, in <module>
    import utils
  File "/content/Proposed/utils.py", line 13, in <module>
    from torch._six import inf
ModuleNotFoundError: No module named 'torch._six'


In [13]:
path = Path('/content/Proposed/utils.py')
code = path.read_text()

old = 'from torch._six import inf'
new = 'from math import inf'

assert old in code, 'No import'
path.write_text(code.replace(old, new))

19091

In [14]:
%cd /content/Proposed
!git diff -- utils.py

/content/Proposed
diff --git a/utils.py b/utils.py
index 42a6edf..dcb5fb5 100644
--- a/utils.py
+++ b/utils.py
@@ -10,7 +10,7 @@ from pathlib import Path
 
 import torch
 import torch.distributed as dist
-from torch._six import inf
+from math import inf
 
 from tensorboardX import SummaryWriter
 


In [15]:
!python main.py --help

Traceback (most recent call last):
  File "/content/Proposed/main.py", line 21, in <module>
    from engine import train_one_epoch, evaluate
  File "/content/Proposed/engine.py", line 7, in <module>
    import utils
  File "/content/Proposed/utils.py", line 15, in <module>
    from tensorboardX import SummaryWriter
ModuleNotFoundError: No module named 'tensorboardX'


In [16]:
%pip install tensorboardX

In [17]:
!python main.py --help

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/content/Proposed/models/convnext.py:156: UserWarning: Overwriting convnext_tiny in registry with models.convnext.convnext_tiny. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/content/Proposed/models/convnext.py:166: UserWarning: Overwriting convnext_small in registry with models.convnext.convnext_small. This is because the name being registered conflicts with an exist

Наконец все исправили, запустим проверку:

In [18]:
%cd /content/Proposed

/content/Proposed


In [20]:
!python main.py \
    --model proposed_base \
    --data_set CIFAR100 \
    --input_size 32 \
    --batch_size 64 \
    --update_freq 4 \
    --lr 4e-3 \
    --drop_path 0.4 \
    --epochs 2 \
    --warmup_epochs 0 \
    --seed 42 \
    --num_workers 2 \
    --data_path /content/cifar_data \
    --output_dir /content/smoke_test \
    --log_dir /content/smoke_logs

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/content/Proposed/models/convnext.py:156: UserWarning: Overwriting convnext_tiny in registry with models.convnext.convnext_tiny. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/content/Proposed/models/convnext.py:166: UserWarning: Overwriting convnext_small in registry with models.convnext.convnext_small. This is because the name being registered conflicts with an exist

In [23]:
path = Path('/content/Proposed/models/Proposed.py')
code = path.read_text()

old = '''def proposed_base(pretrained=False, **kwargs):
    model = Conv2NeXt(depths=[3, 3, 9, 3], dims=[128, 128, 256, 256], dim_power=4, **kwargs)'''
new = '''def proposed_base(pretrained=False, **kwargs):
    kwargs.pop('pretrained_cfg', None)
    kwargs.pop('pretrained_cfg_overlay', None)
    model = Conv2NeXt(depths=[3, 3, 9, 3], dims=[128, 128, 256, 256], dim_power=4, **kwargs)'''

assert old in code, 'No fragment'
path.write_text(code.replace(old, new, 1))

7480

In [24]:
from timm.layers.drop import drop_path
%cd /content/Proposed

from timm.models import create_model
import models.Proposed

model = create_model(
    'proposed_base',
    pretrained=False,
    num_classes=100,
    drop_path_rate=0.4
)

n_params = sum(p.numel() for p in model.parameters())

print('Params:', n_params)

/content/Proposed


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


TypeError: Conv2NeXt.__init__() got an unexpected keyword argument 'cache_dir'

In [28]:
path = Path('/content/Proposed/models/Proposed.py')
code = path.read_text()

old = '''    kwargs.pop('pretrained_cfg_overlay', None)'''
new = '''    kwargs.pop('pretrained_cfg_overlay', None)
    kwargs.pop("cache_dir", None)'''

assert old in code, 'No string'

path.write_text(code.replace(old, new, 1))

7548

In [29]:
%cd /content/Proposed

from timm.models import create_model
import models.Proposed

model = create_model(
    'proposed_base',
    pretrained=False,
    num_classes=100,
    drop_path_rate=0.4
)

n_params = sum(p.numel() for p in model.parameters())

print(f'Params: {n_params / 1e6:.2f} millions')

model.eval()

with torch.no_grad():
    x = torch.randn(2, 3, 32, 32)
    y = model(x)

print('Shape:', y.shape)

/content/Proposed


TypeError: Conv2NeXt.__init__() got an unexpected keyword argument 'cache_dir'

In [30]:
from pathlib import Path

path = Path("/content/Proposed/models/Proposed.py")
code = path.read_text()

code = code.replace(
    '    kwargs.pop("cache_dir", None)\n'
    '    kwargs.pop("cache_dir", None)',
    '    kwargs.pop("cache_dir", None)'
)

path.write_text(code)

7514

In [31]:
import importlib
import models.Proposed

importlib.reload(models.Proposed)

/content/Proposed/models/Proposed.py:180: UserWarning: Overwriting proposed_tiny in registry with models.Proposed.proposed_tiny. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/content/Proposed/models/Proposed.py:186: UserWarning: Overwriting proposed_base in registry with models.Proposed.proposed_base. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model


<module 'models.Proposed' from '/content/Proposed/models/Proposed.py'>

In [32]:
%cd /content/Proposed

from timm.models import create_model
import models.Proposed

model = create_model(
    'proposed_base',
    pretrained=False,
    num_classes=100,
    drop_path_rate=0.4
)

n_params = sum(p.numel() for p in model.parameters())

print(f'Params: {n_params / 1e6:.2f} millions')

model.eval()

with torch.no_grad():
    x = torch.randn(2, 3, 32, 32)
    y = model(x)

print('Shape:', y.shape)

/content/Proposed
Params: 7.41 millions
Shape: torch.Size([2, 100])


In [33]:
%cd /content/Proposed

!python main.py \
    --model proposed_base \
    --data_set CIFAR100 \
    --input_size 32 \
    --batch_size 64 \
    --update_freq 4 \
    --lr 4e-3 \
    --drop_path 0.4 \
    --epochs 2 \
    --warmup_epochs 0 \
    --seed 42 \
    --num_workers 2 \
    --data_path /content/cifar_data \
    --output_dir /content/smoke_test_v2 \
    --log_dir /content/smoke_logs_v2

/content/Proposed
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/content/Proposed/models/convnext.py:156: UserWarning: Overwriting convnext_tiny in registry with models.convnext.convnext_tiny. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/content/Proposed/models/convnext.py:166: UserWarning: Overwriting convnext_small in registry with models.convnext.convnext_small. This is because the name being registered confl

In [34]:
output_dir = Path("/content/smoke_test_v2")

for file in output_dir.rglob("*"):
    if file.is_file():
        print(file.name, f"{file.stat().st_size / 1024**2:.2f} MB")

checkpoint-best.pth 85.26 MB
log.txt 0.00 MB
checkpoint-0.pth 85.26 MB
checkpoint-1.pth 85.26 MB


In [35]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [36]:
import subprocess
import shutil
import sys

dest = Path("/content/drive/MyDrive/convnext_repro")
dest.mkdir(parents=True, exist_ok=True)

subprocess.run(
    ["git", "diff", "--binary"],
    cwd="/content/Proposed",
    stdout=open(dest / "compatibility_changes.patch", "w"),
    check=True
)

commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd="/content/Proposed",
    text=True
).strip()

(dest / "commit.txt").write_text(commit)

with open(dest / "environment.txt", "w") as f:
    subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        stdout=f,
        check=True
    )

shutil.copytree(
    "/content/smoke_test_v2",
    dest / "smoke_test_v2",
    dirs_exist_ok=True
)

PosixPath('/content/drive/MyDrive/convnext_repro/smoke_test_v2')